<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the necessary libraries, load a helper function from `langgraph_utils.py`, and initialize the language models for writing and evaluating content.

`parse_evaluation_response` extracts a quality score (1–10) and feedback from the evaluator's response. `recursion_limit` caps total node executions to prevent infinite loops.

Two models are initialized: `llm_writer` (`gpt-4o-mini`) for drafting and revising, and `llm_evaluator` (`gpt-4o`) for evaluation. Using a stronger model to evaluate a weaker model's output produces genuine quality differences that drive the revision loop naturally.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Local application imports
from langgraph_utils import parse_evaluation_response

# Third-party imports
from langgraph.graph import StateGraph, START, END
from typing import Literal
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
import warnings

warnings.filterwarnings("ignore")

# Initialize the language models
llm_writer = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
llm_evaluator = ChatOpenAI(model="gpt-4o", temperature=0.3)

#### Checkpoint 1/5: Define the Editorial State and Draft Node

Let's define the state schema for our iterative editing pipeline and create an initial draft node that accepts a `state` parameter of type `EditorialState`.

- Create an `EditorialState` class that inherits from `TypedDict` with string fields `topic`, `draft`, and `evaluation`, and integer fields `quality_score` and `iteration`.

- **Drafting:** Define a function `draft_node` that uses `llm_writer` to create a rough blog post (2–3 paragraphs) based on the `topic`, and returns a dictionary with `draft` set to the generated content and `iteration` set to `1`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the state schema for the pipeline
class EditorialState(TypedDict):
    topic: str
    draft: str
    evaluation: str
    quality_score: int
    iteration: int

# Define a node that generates initial draft
def draft_node(state: EditorialState):
    
    result = llm_writer.invoke(
        f"Write a rough draft of a blog post (2-3 paragraphs) on: {state['topic']}."
    )
    
    print(f"Draft created ({len(result.content.split())} words)")
    return {"draft": result.content, "iteration": 1}

#### Checkpoint 2/5: Define the Evaluate and Revise Nodes

With the state and draft node in place, let's create two functions that accept a `state` of type `EditorialState` to evaluate draft quality and revise it based on feedback.

- **Evaluating:** Define a function `evaluate_draft` that uses `llm_evaluator` to assign a quality score (1–10) and provide actionable feedback on the `draft`, then calls `parse_evaluation_response()` to extract `score` and `feedback`. Return a dictionary with `evaluation` (the feedback) and `quality_score` (the score).

- **Revising:** Define a function `revise_draft` that uses `llm_writer` to revise the `draft` based on the `evaluation`, increments the current `iteration` by 1 into `new_iter`, and returns a dictionary with `draft` (the revised content) and `iteration` (the updated count).

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define a node that evaluates the initial draft
def evaluate_draft(state: EditorialState):

    result = llm_evaluator.invoke(
        f"Evaluate this draft. Include a quality score (1-10), and provide actionable feedback on how to improve it.\n\n"
        f"Draft:\n{state['draft']}"
    )
    
    score, feedback = parse_evaluation_response(result.content)
    print(f"Evaluator score: {score}/10 (iteration {state['iteration']})")
    return {"evaluation": feedback, "quality_score": score}

# Define a node that revises the initial draft
def revise_draft(state: EditorialState):

    result = llm_writer.invoke(
        f"Revise this blog draft based on the feedback and produce an improved version.\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Feedback:\n{state['evaluation']}\n"
    )
    
    new_iter = state["iteration"] + 1
    print(f"Draft revised (iteration {new_iter})")
    return {"draft": result.content, "iteration": new_iter}

#### Checkpoint 3/5: Define the Routing Function

Now that both the evaluate and revise nodes are ready, let's create a routing function that decides whether to continue revising or end the loop.

- Define a function `should_revise` that takes a `state` parameter of type `EditorialState` and returns either `"revise"` or `"end"` using a `Literal` type. 

- If `quality_score` is below 8 and `iteration` is under 3, print the score and return `"revise"`. 

- Otherwise, store the reason for stopping (`"quality threshold met"` if the score is 8 or higher, else `"max iterations reached"`) in `reason` and return `"end"`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the routing function
def should_revise(state: EditorialState) -> Literal["revise", "end"]:
    """Decide whether to revise further or exit the loop."""
    
    if state["quality_score"] < 8 and state["iteration"] < 3:
        print(f"Quality score {state['quality_score']}/10 — revising...")
        return "revise"
    
    reason = "quality threshold met" if state["quality_score"] >= 8 else "max iterations reached"
    print(f"Routing done: {reason}")
    return "end"

#### Checkpoint 4/5: Build the Graph with a Cycle

With all nodes and routing logic ready, let's construct the graph with a cycle between evaluation and revision.

- Create a `StateGraph` instance with `EditorialState` as its state type and assign it to `builder`. 

- Add three nodes: `draft` (using `draft_node`), `evaluate` (using `evaluate_draft`), and `revise` (using `revise_draft`). 

- Wire the execution flow with an edge from `START` to `draft`, an edge from `draft` to `evaluate`, a conditional edge from `evaluate` using `should_revise` (mapping `"revise"` to `revise` and `"end"` to `END`), and an edge from `revise` back to `evaluate` to create the cycle.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(EditorialState)

# Add nodes to the graph
builder.add_node("draft", draft_node)
builder.add_node("evaluate", evaluate_draft)
builder.add_node("revise", revise_draft)

# Connect the nodes using edges
builder.add_edge(START, "draft")
builder.add_edge("draft", "evaluate")
builder.add_conditional_edges("evaluate", should_revise, {"revise": "revise", "end": END})
builder.add_edge("revise", "evaluate")  # Cycle

#### Checkpoint 5/5: Compile and Run the Pipeline

Let's compile the pipeline and run it with a sample topic to test the iterative editing process.

- Call `compile()` on `builder` and assign the result to `editorial_pipeline`. 

- Invoke it with a state dictionary containing the `topic` variable, empty strings for `draft` and `evaluation`, `0` for `quality_score` and `iteration`, and a `config` parameter with `recursion_limit` set to `10`. Store the output in `result`.

- Uncomment the `print()` statements to view the final score, iteration count, and draft preview.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Compile the graph into a runnable pipeline
editorial_pipeline = builder.compile()

topic = "How to build a consistent learning routine?"

# Invoke the pipeline with the initial state
result = editorial_pipeline.invoke(
    {
     "topic": topic,
     "draft": "", "evaluation": "",
     "quality_score": 0, "iteration": 0
    },
    config={"recursion_limit": 10}
)

print(f"Final score: {result['quality_score']}/10")
print(f"Iterations: {result['iteration']}")
print(f"\nFinal Draft Preview:\n{result['draft'][:500]}...")

<details> 
    
<summary><strong>Summary</strong></summary> 

In this exercise, we used cycles to allow the pipeline to refine content across multiple passes. A stronger model evaluates each draft, while a lighter model rewrites it based on the feedback — repeating the `evaluate ↔ revise` loop until the content meets the required standard.

</details>